# AnimalPoseTracker × Trimouse：Colab GPU 全链条测试

此 Notebook 使用公开的 `preview` 分支，在 Colab GPU 上完成训练、val 评估、test 预测、TorchScript 导出，以及用导出的模型再次预测。默认训练 10 轮。

运行前请在 Colab 的“更改运行时类型”里选择 GPU，并把 Trimouse 数据文件夹放到 Google Drive 的 `MyDrive/mouse`。如果目录不同，修改下面的 `DRIVE_DATA_ROOT`。

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU。请在 Colab 中选择 GPU 运行时后重试。")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/mouse")
DRIVE_RESULTS_ROOT = Path("/content/drive/MyDrive/AnimalPoseTracker-Colab")

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
REPO_DIR = Path("/content") / f"AnimalPoseTracker-{RUN_ID}"
LOCAL_DATA_ROOT = Path("/content") / f"mouse-data-{RUN_ID}"
PROJECTS_ROOT = Path("/content/animalposetracker-projects")

EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640
NUM_WORKERS = 2
DEVICE = "0"
PREDICT_SPLIT = "test"

print("Run ID:", RUN_ID)
print("Dataset:", DRIVE_DATA_ROOT)
print("Branch:", BRANCH)

In [ ]:
import subprocess

if REPO_DIR.exists():
    raise FileExistsError(f"Clone path already exists: {REPO_DIR}")
subprocess.run(
    ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("Checked out preview commit:", commit)

In [ ]:
import importlib.util
import subprocess
import sys

# Install the repo without desktop GUI dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    cwd=REPO_DIR,
    check=True,
)
module_packages = {
    "cv2": "opencv-contrib-python-headless",
    "yaml": "PyYAML",
    "requests": "requests",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "pycocotools": "pycocotools",
}
missing = [
    package for module, package in module_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        check=True,
    )
print("Installed AnimalPoseTracker from:", REPO_DIR)

In [ ]:
import shutil
import yaml

# Accept either MyDrive/mouse or MyDrive/<folder>/mouse.
source_candidates = [DRIVE_DATA_ROOT, DRIVE_DATA_ROOT / "mouse"]
DATA_SOURCE_ROOT = next(
    (candidate for candidate in source_candidates if (candidate / "images").is_dir()),
    None,
)
if DATA_SOURCE_ROOT is None:
    raise FileNotFoundError(
        f"No images/ directory found under {DRIVE_DATA_ROOT}; edit DRIVE_DATA_ROOT."
    )
if LOCAL_DATA_ROOT.exists():
    raise FileExistsError(f"Local data copy already exists: {LOCAL_DATA_ROOT}")
shutil.copytree(DATA_SOURCE_ROOT, LOCAL_DATA_ROOT)

DATA_YAML = yaml.safe_load(
    (REPO_DIR / "animalposetracker" / "cfg" / "datasets" / "trimouse.yaml")
    .read_text(encoding="utf-8")
)
DATA_YAML["path"] = str(LOCAL_DATA_ROOT)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_counts = {}
for split in ("train", "val", "test"):
    split_dir = LOCAL_DATA_ROOT / "images" / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Missing split directory: {split_dir}")
    image_counts[split] = sum(
        1 for path in split_dir.rglob("*") if path.suffix.lower() in image_extensions
    )
    if image_counts[split] == 0:
        raise ValueError(f"No images found in {split_dir}")

ann_dir = LOCAL_DATA_ROOT / "annotations"
coco_candidates = [
    ann_dir / f"{split}.json" for split in ("train", "val", "test")
] + [
    ann_dir / f"person_keypoints_{split}2017.json"
    for split in ("train", "val", "test")
]
if not DATA_YAML.get("annotation_format"):
    DATA_YAML["annotation_format"] = (
        "coco" if any(path.is_file() for path in coco_candidates) else "yolo"
    )

print("Local dataset:", LOCAL_DATA_ROOT)
print("Annotation format:", DATA_YAML["annotation_format"])
print("Image counts:", image_counts)
print("Keypoint shape:", DATA_YAML["kpt_shape"])
print("Classes:", DATA_YAML["names"])

In [ ]:
from animalposetracker.projector.animalposeproject import AnimalPoseTrackerProject

keypoint_count, keypoint_dims = DATA_YAML["kpt_shape"]
class_names = {
    int(class_id): str(name)
    for class_id, name in DATA_YAML["names"].items()
}
ordered_classes = [class_names[index] for index in sorted(class_names)]
skeleton = DATA_YAML.get("skeleton", [])
keypoint_names = DATA_YAML.get(
    "keypoint_names",
    [f"kpt_{index}" for index in range(keypoint_count)],
)

project = AnimalPoseTrackerProject(
    local_path=PROJECTS_ROOT,
    project_name="trimouse-colab",
    worker="colab",
    model_type="AnimalRTPose",
    model_scale="N",
    keypoints=keypoint_count,
    visible=(keypoint_dims == 3),
    classes=len(class_names),
    keypoints_name=keypoint_names,
    skeleton=skeleton,
    kpt_oks_sigmas=DATA_YAML.get("kpt_oks_sigmas"),
    classes_name=ordered_classes,
    date=RUN_ID,
)
project.create_new_project()
project.dataset_config.update(DATA_YAML)
project.dataset_config["path"] = str(LOCAL_DATA_ROOT)
project.dataset_config["names"] = class_names
project.project_config.update({
    "classes": len(class_names),
    "classes_name": ordered_classes,
    "keypoints": keypoint_count,
    "keypoints_name": keypoint_names,
    "visible": (keypoint_dims == 3),
    "skeleton": skeleton,
    "project_path": str(project.project_path),
})
project._sync_model_config_from_dataset()
project.other_config.update({
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "workers": NUM_WORKERS,
    "device": DEVICE,
    "pretrained": True,
    "val": True,
    "plots": True,
    "project": "runs",
    "name": "train",
    "exist_ok": False,
})
project.save_configs("all")
print("Project:", project.project_path)
print("Pretrained folder:", project.project_path / "pretrained")

In [ ]:
def run_stage(stage_name, start):
    result = start()
    process = project.process
    if process is None:
        raise RuntimeError(f"{stage_name}: no worker process was started")
    return_code = process.wait()
    print(f"{stage_name} exit code:", return_code)
    if return_code != 0:
        raise RuntimeError(f"{stage_name} failed; inspect worker output above")
    return result

run_stage("training", project.train)
run_name = str(project.other_config.get("name", "train"))
run_root = Path(project.other_config.get("project", "runs"))
if not run_root.is_absolute():
    run_root = project.project_path / run_root
train_dir = run_root / run_name
best_checkpoint = train_dir / "weights" / "best.pt"
last_checkpoint = train_dir / "weights" / "last.pt"
if not best_checkpoint.is_file() or not last_checkpoint.is_file():
    raise FileNotFoundError("Training did not produce both best.pt and last.pt")
print("Best checkpoint:", best_checkpoint)
print("Last checkpoint:", last_checkpoint)

In [ ]:
run_stage("independent val evaluation", project.evaluate)
metric_files = sorted(
    (project.project_path / "runs").glob("val*/metrics.json"),
    key=lambda path: path.stat().st_mtime,
)
if not metric_files:
    raise FileNotFoundError("No validation metrics.json was produced")
validation_metrics_path = metric_files[-1]
validation_metrics = json.loads(validation_metrics_path.read_text(encoding="utf-8"))
print("Validation metrics:", validation_metrics_path)
print(json.dumps(validation_metrics, indent=2, ensure_ascii=False))

In [ ]:
native_prediction_dir = run_stage(
    "test prediction from best.pt",
    lambda: project.predict(split=PREDICT_SPLIT),
)
native_predictions_path = native_prediction_dir / "predictions.json"
if not native_predictions_path.is_file():
    raise FileNotFoundError(f"Missing predictions: {native_predictions_path}")
native_predictions = json.loads(native_predictions_path.read_text(encoding="utf-8"))
print("Images predicted from .pt:", len(native_predictions))
print("Prediction folder:", native_prediction_dir)

In [ ]:
run_stage("explicit TorchScript export", lambda: project.export(format="torchscript"))
export_dirs = sorted(
    (project.project_path / "runs").glob("export*"),
    key=lambda path: path.stat().st_mtime,
)
if not export_dirs:
    raise FileNotFoundError("No export directory was produced")
torchscript_model = export_dirs[-1] / "best.torchscript"
if not torchscript_model.is_file():
    raise FileNotFoundError(f"Missing TorchScript model: {torchscript_model}")
print("TorchScript model:", torchscript_model)

native_model_setting = project.other_config.get("model")
project.other_config["model"] = str(torchscript_model)
project.save_configs("other")
torchscript_prediction_dir = run_stage(
    "test prediction from TorchScript",
    lambda: project.predict(split=PREDICT_SPLIT),
)
torchscript_predictions_path = torchscript_prediction_dir / "predictions.json"
if not torchscript_predictions_path.is_file():
    raise FileNotFoundError(f"Missing exported-model predictions: {torchscript_predictions_path}")
torchscript_predictions = json.loads(torchscript_predictions_path.read_text(encoding="utf-8"))
print("Images predicted from TorchScript:", len(torchscript_predictions))

project.other_config["model"] = native_model_setting
project.save_configs("other")

In [ ]:
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_RESULTS_ROOT / project.project_path.name
if RESULTS_DIR.exists():
    raise FileExistsError(f"Results already exist: {RESULTS_DIR}")
RESULTS_DIR.mkdir(parents=True)
shutil.copytree(project.project_path / "runs", RESULTS_DIR / "runs")
shutil.copytree(project.project_path / "pretrained", RESULTS_DIR / "pretrained")
shutil.copytree(project.project_path / "configs", RESULTS_DIR / "configs")
shutil.copy2(project.project_path / "project.yaml", RESULTS_DIR / "project.yaml")

# Store the original Drive data location in this results snapshot.
dataset_snapshot_path = RESULTS_DIR / "configs" / "dataset.yaml"
dataset_snapshot = yaml.safe_load(dataset_snapshot_path.read_text(encoding="utf-8"))
dataset_snapshot["path"] = str(DATA_SOURCE_ROOT)
dataset_snapshot_path.write_text(
    yaml.safe_dump(dataset_snapshot, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)
other_snapshot_path = RESULTS_DIR / "configs" / "other.yaml"
other_snapshot = yaml.safe_load(other_snapshot_path.read_text(encoding="utf-8"))
other_snapshot["model"] = "configs/model.yaml"
other_snapshot["data"] = "configs/dataset.yaml"
def rewrite_pretrained_path(node):
    if isinstance(node, dict):
        for key, value in list(node.items()):
            if key == "pretrained" and isinstance(value, str) and value.startswith("/content/"):
                node[key] = "pretrained/animalrtpose-n.pt"
            else:
                rewrite_pretrained_path(value)
    elif isinstance(node, list):
        for value in node:
            rewrite_pretrained_path(value)

rewrite_pretrained_path(other_snapshot)
other_snapshot_path.write_text(
    yaml.safe_dump(other_snapshot, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

summary = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": commit,
    "model": "AnimalRTPose-N",
    "dataset_root": str(DATA_SOURCE_ROOT),
    "annotation_format": DATA_YAML["annotation_format"],
    "image_counts": image_counts,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "validation_metrics": str(validation_metrics_path),
    "native_prediction_dir": str(native_prediction_dir),
    "torchscript_model": str(torchscript_model),
    "torchscript_prediction_dir": str(torchscript_prediction_dir),
    "saved_results": str(RESULTS_DIR),
}
(RESULTS_DIR / "colab_run_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Saved outputs to:", RESULTS_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))